### 데이터 로드

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
train_cn7 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv")
train_rg3 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_rg3.csv")

### X / Y 분리

In [2]:
TARGET = "PassOrFail"

X = train_cn7.drop(columns=[TARGET])
y = train_cn7[TARGET]

print(X.shape)
print(y.value_counts())
print(y.value_counts(normalize=True))

(1211, 25)
PassOrFail
0    1194
1      17
Name: count, dtype: int64
PassOrFail
0    0.985962
1    0.014038
Name: proportion, dtype: float64


### 상수 컬럼 제거

In [3]:
constant_cols = X.columns[X.nunique() <= 1].tolist()

print("상수 컬럼:", constant_cols)

X = X.drop(columns=constant_cols)

상수 컬럼: ['Clamp_Open_Position']


### 데이터 분리 
> stratify = y

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train")
print(y_train.value_counts())

print("\nTest")
print(y_test.value_counts())

Train
PassOrFail
0    954
1     14
Name: count, dtype: int64

Test
PassOrFail
0    240
1      3
Name: count, dtype: int64


In [5]:
from imblearn.over_sampling import RandomOverSampler

ratios = [0.1, 0.2, 0.3, 0.5, 1.0]

for ratio in ratios:
    ros = RandomOverSampler(
        sampling_strategy=ratio,
        random_state=42
    )

    X_train_ros, y_train_ros = ros.fit_resample(
        X_train,
        y_train
    )

    print(f"\n[Ratio {ratio}]")
    print(y_train_ros.value_counts())


[Ratio 0.1]
PassOrFail
0    954
1     95
Name: count, dtype: int64

[Ratio 0.2]
PassOrFail
0    954
1    190
Name: count, dtype: int64

[Ratio 0.3]
PassOrFail
0    954
1    286
Name: count, dtype: int64

[Ratio 0.5]
PassOrFail
0    954
1    477
Name: count, dtype: int64

[Ratio 1.0]
PassOrFail
0    954
1    954
Name: count, dtype: int64


### baseline logistic regression

In [11]:
from sklearn.linear_model import LogisticRegression

# 오버샘플링 후 Logistic Regression 학습
lr_ros = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_ros.fit(X_train_ros, y_train_ros)

# 예측
y_pred_ros = lr_ros.predict(X_test)
y_prob_ros = lr_ros.predict_proba(X_test)[:, 1]

### 평가

In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

lr_ros.fit(X_train, y_train)

y_pred_balanced = lr_ros.predict(X_test)
y_prob_balanced = lr_ros.predict_proba(X_test)[:, 1]

print("[CN7 Logistic Regression + oversampling]")
print("Accuracy :", accuracy_score(y_test, y_pred_balanced))
print("Precision:", precision_score(y_test, y_pred_balanced, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred_balanced, zero_division=0))
print("F1       :", f1_score(y_test, y_pred_balanced, zero_division=0))
print("PR-AUC   :", average_precision_score(y_test, y_pred_balanced))

[CN7 Logistic Regression + oversampling]
Accuracy : 0.9876543209876543
Precision: 0.0
Recall   : 0.0
F1       : 0.0
PR-AUC   : 0.012345679012345678


In [13]:
print("불량 비율:", y_test.mean())
print("PR-AUC:", average_precision_score(y_test, y_prob_ros))

print("\n실제 불량 예측확률")
print(y_prob_ros[y_test.values == 1])

print("\n예측확률 범위")
print("min :", y_prob_ros.min())
print("max :", y_prob_ros.max())

불량 비율: 0.012345679012345678
PR-AUC: 0.11869747899159663

실제 불량 예측확률
[0.6567763  0.20988853 0.01296099]

예측확률 범위
min : 1.3497388139547902e-10
max : 0.9816835370893957


In [14]:
print("불량 비율:", y_test.mean())
print("PR-AUC:", average_precision_score(y_test, y_prob_ros))

print("\n실제 불량 3개의 예측확률")
print(y_prob_ros[y_test.values == 1])

print("\n전체 예측확률 범위")
print("min:", y_prob_ros.min())
print("max:", y_prob_ros.max())

불량 비율: 0.012345679012345678
PR-AUC: 0.11869747899159663

실제 불량 3개의 예측확률
[0.6567763  0.20988853 0.01296099]

전체 예측확률 범위
min: 1.3497388139547902e-10
max: 0.9816835370893957


In [15]:
result = pd.DataFrame({
    "actual": y_test.values,
    "prob": y_prob_ros
})

result = result.sort_values(
    "prob",
    ascending=False
).reset_index(drop=True)

result["rank"] = result.index + 1

print(result[result["actual"] == 1])

    actual      prob  rank
7        1  0.656776     8
13       1  0.209889    14
33       1  0.012961    34


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Logistic Regression
lr_ros = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Oversampling된 Train으로 학습
lr_ros.fit(X_train_ros, y_train_ros)

# 원본 Test 평가
y_pred_ros = lr_ros.predict(X_test)
y_prob_ros = lr_ros.predict_proba(X_test)[:, 1]

print("[CN7 Logistic Regression + RandomOverSampler]")
print("Accuracy :", accuracy_score(y_test, y_pred_ros))
print("Precision:", precision_score(y_test, y_pred_ros, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred_ros, zero_division=0))
print("F1       :", f1_score(y_test, y_pred_ros, zero_division=0))
print("PR-AUC   :", average_precision_score(y_test, y_prob_ros))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_ros))

print("\nClassification Report")
print(
    classification_report(
        y_test,
        y_pred_ros,
        digits=4,
        zero_division=0
    )
)

[CN7 Logistic Regression + RandomOverSampler]
Accuracy : 0.9547325102880658
Precision: 0.1
Recall   : 0.3333333333333333
F1       : 0.15384615384615385
PR-AUC   : 0.11869747899159663

Confusion Matrix
[[231   9]
 [  2   1]]

Classification Report
              precision    recall  f1-score   support

           0     0.9914    0.9625    0.9767       240
           1     0.1000    0.3333    0.1538         3

    accuracy                         0.9547       243
   macro avg     0.5457    0.6479    0.5653       243
weighted avg     0.9804    0.9547    0.9666       243

